# s04 step by step: from GRAPH files to a trained model

This notebook follows `scripts/python/s04_train_model.py` function by function, on **tiny toy data**, so you can look at
every intermediate object: its type, its dtype, its shape and its values. Nothing here touches the real data on scratch.

The functions below are copies of the ones in s04 (same names, same logic). Each section first runs a function, then
prints what came out. Change the toy sizes, rerun cells, break things on purpose — that's what this notebook is for.

| Section | s04 function(s) | what you look at |
|---|---|---|
| 1 | (s03 output) | the GRAPH parquet files: columns, dtypes, pools |
| 2 | `build_pool_index`, `sample_pools` | the list of pools and the random sample |
| 3 | `load_pools`, `load_norm_stats` | one pool as a dict of numpy arrays |
| 4 | `draw_subset` … `build_edge_features` | building **one** graph by hand |
| 5 | `PoolDataset`, `estimate_edge_stats` | the same, packed into a class; fresh vs fixed draws |
| 6 | `DataLoader` | how several graphs become one batch |
| 7 | `InSARGNN` | the shapes inside the model |
| 8 | `loss_fn` | what the NLL rewards and punishes |
| 9 | `train_one_epoch`, `evaluate`, `train_model`, `evaluate_on_dev` | training, early stopping, checkpoint, metrics |

Kernel: **LOSTInPLACE Tutorial** (this folder's uv environment, see `README.md`).

## 0. Setup

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from pyproj import Geod
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import to_undirected

# geodesic distances and bearings on the WGS84 ellipsoid, as in s04
GEOD = Geod(ellps="WGS84")

torch.manual_seed(0)
print("torch", torch.__version__, "| polars", pl.__version__, "| cuda available:", torch.cuda.is_available())

## 1. Toy GRAPH files (what s03 writes)

s03 writes one `<burst>_GRAPH.parquet` per burst into `data/<set>/<location>/<orbit>/`. Every row is one EGMS point that
belongs to a pool. A pool = all rows with the same `seed_pid`; exactly one row of it has `seed == 1` (the seed).

We build the same structure in a temporary folder: 2 train regions and 1 dev region, a few pools each, with a smooth
fake velocity field (so the model has something learnable) and random fake GSE vectors.

In [ ]:
ROOT = Path(tempfile.mkdtemp()) / "data"
rng = np.random.default_rng(0)

def toy_velocity(lat, lon):
    # a smooth fake deformation field in mm/yr: a subsidence bowl plus a gentle trend
    return -15 * np.exp(-((lat - 48.10) ** 2 + (lon - 11.50) ** 2) / 0.0004) + 20 * (lon - 11.5)

def quantise_gse(x):
    # the inverse of the de-quantisation in s04: random unit vectors -> int8, like the real GSE columns
    return np.clip(np.round(np.sign(x) * np.sqrt(np.abs(x)) * 127.5), -127, 127).astype(np.int8)

def make_graph_file(path, n_pools, centre, prefix):
    rows = []
    for p in range(n_pools):
        n = int(rng.integers(5, 27))                      # pool size incl. the seed (some below min_graph_size = 7)
        seed_lat = centre[0] + rng.normal(0, 0.01); seed_lon = centre[1] + rng.normal(0, 0.01)
        lat = seed_lat + np.r_[0, rng.normal(0, 0.005, n - 1)]   # first point = the seed, the others within ~1 km
        lon = seed_lon + np.r_[0, rng.normal(0, 0.007, n - 1)]
        gse = rng.normal(size=(n, 64)); gse /= np.linalg.norm(gse, axis=1, keepdims=True)
        seed_pid = f"{prefix}_{p}_0"
        for j in range(n):
            row = {"pid": f"{prefix}_{p}_{j}", "seed_pid": seed_pid, "seed": np.int8(j == 0),
                   "latitude": lat[j], "longitude": lon[j], "easting": 0.0, "northing": 0.0,
                   "incidence_angle": 38 + rng.normal(0, 3),
                   "mean_velocity": toy_velocity(lat[j], lon[j]) + rng.normal(0, 1)}
            row.update({f"gse_{i}": v for i, v in enumerate(quantise_gse(gse[j]))})
            rows.append(row)
    df = pl.DataFrame(rows).with_columns(pl.col("seed").cast(pl.Int8), pl.col([f"gse_{i}" for i in range(64)]).cast(pl.Int8))
    path.parent.mkdir(parents=True, exist_ok=True)
    df.write_parquet(path)
    return str(path)

train_files = [make_graph_file(ROOT / "train" / "RegionA" / "ascending" / "burstA1_GRAPH.parquet", 12, (48.10, 11.50), "A1"),
               make_graph_file(ROOT / "train" / "RegionA" / "descending" / "burstA2_GRAPH.parquet", 10, (48.10, 11.50), "A2"),
               make_graph_file(ROOT / "train" / "RegionB" / "ascending" / "burstB1_GRAPH.parquet", 12, (48.12, 11.47), "B1")]
dev_files = [make_graph_file(ROOT / "dev" / "RegionC" / "ascending" / "burstC1_GRAPH.parquet", 8, (48.11, 11.49), "C1")]
print("train files:", *train_files, sep="\n  ")
print("dev files:", *dev_files, sep="\n  ")

In [ ]:
# look at one GRAPH file: columns, dtypes, number of rows and pools
g = pl.read_parquet(train_files[0])
print("shape (rows, columns):", g.shape)
print("schema (first 9 columns):", dict(list(g.schema.items())[:9]))
print("GSE column dtype:", g["gse_0"].dtype)
print("pools in this file:", g["seed_pid"].n_unique(), "| seeds:", int(g["seed"].sum()))
g.select("pid", "seed_pid", "seed", "latitude", "longitude", "incidence_angle", "mean_velocity", "gse_0", "gse_1").head(6)

In [ ]:
# points per pool: the pool sizes vary (sparse areas give smaller pools)
g.group_by("seed_pid").len().sort("seed_pid")

## 2. The pool index and the sample — `build_pool_index`, `sample_pools`

Instead of loading all points, s04 first only lists the pools: one row per pool, (`seed_pid`, `graph_file`). Because every
pool has exactly one seed row, filtering `seed == 1` gives exactly one row per pool. The sample (`--n-graphs`) is then
drawn from this cheap list.

In [ ]:
def build_pool_index(graph_files):
    tables = [pl.scan_parquet(path).filter(pl.col("seed") == 1).select("seed_pid")
              .with_columns(pl.lit(str(path)).alias("graph_file")).collect() for path in graph_files]
    return pl.concat(tables)

def sample_pools(pool_index, n_graphs, seed):
    if n_graphs is None or n_graphs >= pool_index.height:
        return pool_index
    rng = np.random.default_rng(seed)
    positions = rng.choice(pool_index.height, size=n_graphs, replace=False)
    return pool_index[positions]

train_index_all = build_pool_index(train_files)
print(type(train_index_all), train_index_all.shape)
train_index_all.head()

In [ ]:
# --n-graphs 15: 15 of the train pools, drawn over all files at once
train_index = sample_pools(train_index_all, n_graphs=15, seed=42)
print("sampled pools per file:")
print(train_index.group_by("graph_file").len().with_columns(pl.col("graph_file").str.split("/").list.slice(-3).list.join("/")))
# same seed -> same sample, different seed -> different sample
print("same seed reproduces:", sample_pools(train_index_all, 15, 42).equals(train_index))

## 3. Loading the pools — `load_pools`, `load_norm_stats`

Now the points of the sampled pools are read, every file only once (`group_by("graph_file")`). Each pool becomes a dict of
numpy arrays. Pools below `min_graph_size` are dropped here.

In [ ]:
def load_pools(sampled_index, use_gse, min_graph_size):
    cols = ["latitude", "longitude", "mean_velocity", "incidence_angle", "seed_pid"]
    gse_cols = [f"gse_{i}" for i in range(64)]
    if use_gse:
        cols += gse_cols
    pools = []
    for (graph_file,), rows in sampled_index.group_by("graph_file"):
        df = pl.read_parquet(graph_file, columns=cols)
        df = df.filter(pl.col("seed_pid").is_in(rows["seed_pid"].implode()))
        for pool_df in df.partition_by("seed_pid", as_dict=True).values():
            if pool_df.height < min_graph_size:
                continue
            pool = {"lat": pool_df["latitude"].to_numpy(), "lon": pool_df["longitude"].to_numpy(),
                    "vel": pool_df["mean_velocity"].to_numpy().astype(np.float32),
                    "inc": pool_df["incidence_angle"].to_numpy().astype(np.float32),
                    "region": Path(graph_file).parent.parent.name, "orbit": Path(graph_file).parent.name}
            if use_gse:
                pool["gse"] = pool_df.select(gse_cols).to_numpy()
            pools.append(pool)
    return pools

USE_GSE, MIN_GRAPH_SIZE = True, 7
train_pools = load_pools(train_index, USE_GSE, MIN_GRAPH_SIZE)
print("sampled:", train_index.height, "| kept (>= min_graph_size):", len(train_pools))

pool = train_pools[0]
print("\none pool is a", type(pool), "with keys", list(pool))
for k, v in pool.items():
    print(f"  {k:7s}", type(v).__name__, getattr(v, "dtype", ""), getattr(v, "shape", v))

In [ ]:
# the normalization stats: in s04 they come from assets/egms_norm_stats.json (train regions, written by s02).
# here we compute toy stats from the toy train pools in the same format
all_vel = np.concatenate([p["vel"] for p in train_pools]); all_inc = np.concatenate([p["inc"] for p in train_pools])
stats = {"mean_velocity": {"mean": float(all_vel.mean()), "std": float(all_vel.std())},
         "incidence_angle": {"mean": float(all_inc.mean()), "std": float(all_inc.std())}}
print(json.dumps(stats, indent=2))

## 4. One graph by hand — what `PoolDataset.__getitem__` does

This is the core of s04: from one pool, a **random subset** is drawn, one point of it becomes the **query** (its velocity is
hidden), the points are connected to their **k nearest neighbours**, and the node and edge features are built.
We go through it one function at a time.

### 4.1 `draw_subset` — which points, which query

In [ ]:
def draw_subset(n, subset_min, subset_max, min_graph_size, rng):
    share = rng.uniform(subset_min, subset_max)
    m = min(n, max(min_graph_size, round(share * n)))
    idx = rng.choice(n, size=m, replace=False)
    q = rng.integers(m)
    return idx, q

n = len(pool["vel"])
idx, q = draw_subset(n, 0.6, 0.9, MIN_GRAPH_SIZE, np.random.default_rng(1))
print("pool size n =", n, "| subset size m =", len(idx))
print("idx (positions in the pool):", idx, idx.dtype)
print("q =", q, "-> the query is pool point", idx[q])

### 4.2 `pairwise_geodesic` — all distances and bearings

In [ ]:
def pairwise_geodesic(lat, lon):
    m = len(lat)
    lat1, lon1 = np.repeat(lat, m), np.repeat(lon, m)
    lat2, lon2 = np.tile(lat, m), np.tile(lon, m)
    az12, _, d = GEOD.inv(lon1, lat1, lon2, lat2)
    return d.reshape(m, m), az12.reshape(m, m)

lat, lon = pool["lat"][idx], pool["lon"][idx]
print("repeat vs tile for m = 3:", np.repeat([1, 2, 3], 3), np.tile([1, 2, 3], 3), "-> every (i, j) pair once")
dist, az = pairwise_geodesic(lat, lon)
print("dist", dist.shape, dist.dtype, "| az", az.shape)
np.set_printoptions(precision=0, suppress=True, linewidth=150)
print("distances (m), first 5 x 5:\n", dist[:5, :5])
print("bearings (deg), first 5 x 5: row i -> column j\n", az[:5, :5])
print("symmetric distance:", np.allclose(dist, dist.T), "| bearing i->j and j->i differ by ~180:", round(az[0, 1]), round(az[1, 0]))

### 4.3 `knn_edges` — who is connected to whom

In [ ]:
def knn_edges(dist, k):
    m = len(dist)
    dist = dist.copy()
    np.fill_diagonal(dist, np.inf)
    neighbours = np.argsort(dist, axis=1)[:, :k]
    target = np.repeat(np.arange(m), k)
    source = neighbours.reshape(-1)
    edge_index = torch.tensor(np.stack([source, target]), dtype=torch.long)
    return to_undirected(edge_index), torch.tensor(np.stack([source, target]), dtype=torch.long), neighbours

K = 3
edge_index, directed, neighbours = knn_edges(dist, K)
print("3 nearest neighbours of every node (row = node):\n", neighbours[:5], "...")
print("\ndirected edges (neighbour -> node), shape", tuple(directed.shape), "\n", directed[:, :9], "...")
print("\nafter to_undirected, shape", tuple(edge_index.shape), "dtype", edge_index.dtype)
print("every node gets at least k incoming edges:", torch.bincount(edge_index[1]).min().item() >= K)
print("in-degree per node:", torch.bincount(edge_index[1]).tolist())

### 4.4 `build_node_features` and `dequantise_gse` — the matrix `x`

In [ ]:
def dequantise_gse(v):
    v = v.astype(np.float32)
    return (v / 127.5) ** 2 * np.sign(v)

def build_node_features(pool, idx, q, stats, use_gse):
    m = len(idx)
    v_stats, i_stats = stats["mean_velocity"], stats["incidence_angle"]
    vel = (pool["vel"][idx] - v_stats["mean"]) / v_stats["std"]
    vel[q] = 0.0
    is_query = np.zeros(m, dtype=np.float32); is_query[q] = 1.0
    inc = (pool["inc"][idx] - i_stats["mean"]) / i_stats["std"]
    gse = dequantise_gse(pool["gse"][idx]) if use_gse else None
    x = np.column_stack([vel, is_query, inc] + ([gse] if use_gse else []))
    return torch.tensor(x, dtype=torch.float32)

np.set_printoptions(precision=3, suppress=True)
raw = pool["gse"][idx[:2], :6]
print("GSE raw int8:\n", raw, "\nde-quantised:\n", np.round(dequantise_gse(raw), 3))
print("vector length after de-quantisation (should be ~1):", np.round(np.linalg.norm(dequantise_gse(pool["gse"][idx]), axis=1)[:5], 3))

x = build_node_features(pool, idx, q, stats, use_gse=True)
x_no_gse = build_node_features(pool, idx, q, stats, use_gse=False)
print("\nx with GSE:", tuple(x.shape), x.dtype, "| without GSE:", tuple(x_no_gse.shape))
print("columns [vel, is_query, inc]; the query row has vel = 0 and is_query = 1:")
print(x_no_gse.numpy())

### 4.5 `build_edge_features` — one row per edge

In [ ]:
def build_edge_features(dist, az, edge_index, stats):
    src, dst = edge_index.numpy()
    d = np.log1p(dist[src, dst])
    if "edge_dist" in stats:
        d = (d - stats["edge_dist"]["mean"]) / stats["edge_dist"]["std"]
    bearing = np.radians(az[dst, src])
    return torch.tensor(np.column_stack([d, np.sin(bearing), np.cos(bearing)]), dtype=torch.float32)

edge_attr = build_edge_features(dist, az, edge_index, stats)   # no "edge_dist" in stats yet -> raw log(1 + d)
print("edge_attr", tuple(edge_attr.shape), "= (number of edges, 3) -> [log distance, sin, cos]")
print("first edges: source -> target, then their features")
for e in range(4):
    s_, t_ = edge_index[:, e].tolist()
    print(f"  {s_:2d} -> {t_:2d}  dist {dist[s_, t_]:7.1f} m  features {edge_attr[e].numpy()}")
print("sin^2 + cos^2 = 1:", torch.allclose(edge_attr[:, 1] ** 2 + edge_attr[:, 2] ** 2, torch.ones(len(edge_attr))))

### 4.6 Target, query mask and the finished `Data` object

In [ ]:
y = torch.tensor([(pool["vel"][idx][q] - stats["mean_velocity"]["mean"]) / stats["mean_velocity"]["std"]], dtype=torch.float32)
query_mask = torch.zeros(len(idx), dtype=torch.bool); query_mask[q] = True
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, query_mask=query_mask, region=pool["region"], orbit=pool["orbit"])
print(data)
print("y (normalized target):", y, "| true velocity of the query (mm/yr):", pool["vel"][idx][q])
print("query_mask:", query_mask.int().tolist())
print("the hidden value is NOT in x:", x[q, 0].item(), "(only in y)")

## 5. Packed into a class — `PoolDataset`, `estimate_edge_stats`

`PoolDataset` stores the pools and settings; `__getitem__(i)` runs exactly the steps of section 4. Two calls on the same
pool give **different** graphs in train mode, **identical** ones with `fixed_draws=True` (dev).

In [ ]:
class PoolDataset(torch.utils.data.Dataset):
    def __init__(self, pools, stats, k_graph, subset_min, subset_max, min_graph_size, use_gse, fixed_draws=False):
        self.pools, self.stats, self.k_graph = pools, stats, k_graph
        self.subset_min, self.subset_max, self.min_graph_size = subset_min, subset_max, min_graph_size
        self.use_gse, self.fixed_draws = use_gse, fixed_draws

    def __len__(self):
        return len(self.pools)

    def __getitem__(self, i):
        pool = self.pools[i]
        n = len(pool["vel"])
        rng = np.random.default_rng(i) if self.fixed_draws else np.random.default_rng()
        idx, q = draw_subset(n, self.subset_min, self.subset_max, self.min_graph_size, rng)
        dist, az = pairwise_geodesic(pool["lat"][idx], pool["lon"][idx])
        edge_index = knn_edges(dist, self.k_graph)[0]          # [0]: the notebook version also returns extras
        x = build_node_features(pool, idx, q, self.stats, self.use_gse)
        edge_attr = build_edge_features(dist, az, edge_index, self.stats)
        v = self.stats["mean_velocity"]
        y = torch.tensor([(pool["vel"][idx][q] - v["mean"]) / v["std"]], dtype=torch.float32)
        query_mask = torch.zeros(len(idx), dtype=torch.bool); query_mask[q] = True
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, query_mask=query_mask,
                    region=pool["region"], orbit=pool["orbit"])

train_ds = PoolDataset(train_pools, stats, K, 0.6, 0.9, MIN_GRAPH_SIZE, USE_GSE)
print("len(train_ds) =", len(train_ds), "-> one item per pool")
a, b = train_ds[0], train_ds[0]
print("train, same pool twice:", a.num_nodes, "vs", b.num_nodes, "nodes | same y:", torch.equal(a.y, b.y))

dev_pools = load_pools(build_pool_index(dev_files), USE_GSE, MIN_GRAPH_SIZE)
dev_ds = PoolDataset(dev_pools, stats, K, 0.6, 0.9, MIN_GRAPH_SIZE, USE_GSE, fixed_draws=True)
a, b = dev_ds[0], dev_ds[0]
print("dev (fixed_draws), same pool twice: identical x and edges:", torch.equal(a.x, b.x) and torch.equal(a.edge_index, b.edge_index))

In [ ]:
def estimate_edge_stats(dataset, n_draws=2000):
    graphs = [dataset[i] for i in np.random.default_rng(0).integers(0, len(dataset), n_draws)]
    d = torch.cat([g.edge_attr[:, 0] for g in graphs]).numpy()      # raw log(1 + d) of all edges of all draws
    return {**dataset.stats, "edge_dist": {"mean": float(d.mean()), "std": float(d.std())}}

stats = estimate_edge_stats(train_ds)
train_ds.stats = dev_ds.stats = stats      # from now on the edge distance is normalized as well
print(json.dumps(stats["edge_dist"], indent=2))
print("edge distance column now:", train_ds[0].edge_attr[:5, 0].numpy(), "(mean ~0, std ~1 over many draws)")

## 6. Batching — what the `DataLoader` makes of several graphs

PyG combines a batch of graphs into **one big disconnected graph**: the node tables are stacked, the edge indices are
shifted so they point to the right rows, and `batch.batch` says which graph every node belongs to. Strings like `region`
become lists. Because every graph has exactly one query, `query_mask` selects exactly one node per graph.

In [ ]:
loader = DataLoader(train_ds, batch_size=4, shuffle=True)
batch = next(iter(loader))
print(batch)
print("num_graphs:", batch.num_graphs, "| nodes per graph:", torch.bincount(batch.batch).tolist(), "| total nodes:", batch.num_nodes)
print("ptr (first node of every graph):", batch.ptr.tolist())
print("batch.batch (graph id of every node):", batch.batch.tolist())
print("x:", tuple(batch.x.shape), "| edge_index:", tuple(batch.edge_index.shape), "| edge_attr:", tuple(batch.edge_attr.shape))
print("y:", tuple(batch.y.shape), "-> one target per graph | queries selected by query_mask:", int(batch.query_mask.sum()))
print("region / orbit (lists):", batch.region, batch.orbit)

In [ ]:
# the edge indices of graph 1 are shifted by the number of nodes of graph 0 (= ptr[1])
g1_edges = batch.edge_index[:, batch.batch[batch.edge_index[0]] == 1]
print("edges of graph 1 point to rows", g1_edges.min().item(), "...", g1_edges.max().item(),
      "| its nodes are rows", batch.ptr[1].item(), "...", batch.ptr[2].item() - 1)

## 7. The model — `InSARGNN`

The model sees the whole batch at once. Follow the shapes: `(nodes, features)` → `(nodes, hidden)` through the encoder and
the message passing layers → `(graphs, hidden)` at the query nodes → one `mu` and one `logvar` per graph.

In [ ]:
class InSARGNN(nn.Module):
    def __init__(self, n_node_features, edge_dim, hidden_dim, n_layers, heads, dropout):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_node_features, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        self.convs = nn.ModuleList([GATv2Conv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=edge_dim, dropout=dropout)
                                    for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.head_mu = nn.Linear(hidden_dim, 1)
        self.head_logvar = nn.Linear(hidden_dim, 1)

    def forward(self, data, verbose=False):
        h = self.encoder(data.x)
        if verbose: print("  after encoder:        ", tuple(h.shape))
        for layer, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            h = norm(h + nn.functional.gelu(conv(h, data.edge_index, data.edge_attr)))
            if verbose: print(f"  after message pass {layer}:", tuple(h.shape))
        h_query = h[data.query_mask]
        if verbose: print("  query nodes only:     ", tuple(h_query.shape))
        mu = self.head_mu(h_query).squeeze(-1)
        logvar = self.head_logvar(h_query).squeeze(-1).clamp(-8, 8)
        return mu, logvar

n_node_features, edge_dim = train_ds[0].x.shape[1], train_ds[0].edge_attr.shape[1]
model = InSARGNN(n_node_features, edge_dim, hidden_dim=32, n_layers=2, heads=4, dropout=0.1)
print(model)
print("\ntrainable parameters:", sum(p.numel() for p in model.parameters()))
print("\ninput x:", tuple(batch.x.shape))
mu, logvar = model(batch, verbose=True)
print("mu:", tuple(mu.shape), mu.detach().numpy(), "\nlogvar:", tuple(logvar.shape), logvar.detach().numpy())

In [ ]:
# what nn.Module registers for you: these are the weights the optimizer trains and state_dict() saves
for name, p in list(model.named_parameters())[:6]:
    print(f"{name:35s} {tuple(p.shape)}")
print("...", len(list(model.parameters())), "parameter tensors in total")

## 8. The loss — `loss_fn`

Heteroscedastic Gaussian NLL: `0.5 * (logvar + (y - mu)^2 / exp(logvar))`. A large error is cheaper if the model also says
"I'm unsure" (large `logvar`), but being unsure costs `logvar` itself. So the model learns an honest uncertainty.

In [ ]:
def loss_fn(mu, logvar, y):
    return (0.5 * (logvar + (y - mu) ** 2 / torch.exp(logvar))).mean()

y = torch.tensor([1.0])
for mu_, lv in [(1.0, 0.0), (1.0, -3.0), (0.0, 0.0), (0.0, -3.0), (0.0, 1.0), (0.0, 3.0)]:
    print(f"error {abs(1 - mu_):.0f}, logvar {lv:4.1f} (sigma {np.exp(lv / 2):5.2f}) -> loss {loss_fn(torch.tensor([mu_]), torch.tensor([lv]), y).item():6.3f}")
print("-> right and confident is best; wrong and confident is worst; wrong but unsure is in between")

## 9. Training, early stopping, checkpoint, metrics

`train_one_epoch` and `evaluate` as in s04, then `train_model`'s epoch loop with early stopping on the dev loss and a
checkpoint at every new best epoch. On toy data a few epochs are enough to see the mechanics.

In [ ]:
def train_one_epoch(model, loader, optimizer, grad_clip, device):
    model.train()
    total_loss, n_graphs = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        mu, logvar = model(batch)
        loss = loss_fn(mu, logvar, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        n_graphs += batch.num_graphs
    return total_loss / n_graphs

def evaluate(model, loader, device):
    model.eval()
    with torch.no_grad():
        mus, logvars, ys, regions, orbits = [], [], [], [], []
        for batch in loader:
            batch = batch.to(device)
            mu, logvar = model(batch)
            mus.append(mu.cpu()); logvars.append(logvar.cpu()); ys.append(batch.y.cpu())
            regions.extend(batch.region); orbits.extend(batch.orbit)
        mu, logvar, y = torch.cat(mus), torch.cat(logvars), torch.cat(ys)
        nll = loss_fn(mu, logvar, y).item()
        v = loader.dataset.stats["mean_velocity"]
        mu_mm, y_mm, sigma_mm = mu * v["std"] + v["mean"], y * v["std"] + v["mean"], torch.exp(0.5 * logvar) * v["std"]
        err = mu_mm - y_mm
        metrics = {"nll": nll, "rmse": torch.sqrt((err ** 2).mean()).item(), "mae": err.abs().mean().item(),
                   "inside_90": (err.abs() <= 1.645 * sigma_mm).float().mean().item()}
        predictions = pl.DataFrame({"region": regions, "orbit": orbits, "y_mm": y_mm.numpy(),
                                    "mu_mm": mu_mm.numpy(), "sigma_mm": sigma_mm.numpy()})
    return metrics, predictions

device = "cuda" if torch.cuda.is_available() else "cpu"
model = InSARGNN(n_node_features, edge_dim, hidden_dim=32, n_layers=2, heads=4, dropout=0.1).to(device)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=4, shuffle=False)

m, p = evaluate(model, dev_loader, device)
print("untrained model on dev:", {k: round(v, 3) for k, v in m.items()})
p

In [ ]:
# train_model: the epoch loop with early stopping (small patience so you can see it stop)
checkpoint_path = Path(tempfile.mkdtemp()) / "checkpoints" / "toy_run" / "best_model.pt"
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
best_loss, best_epoch, epochs_without_improvement, history = float("inf"), None, 0, []
MAX_EPOCHS, PATIENCE = 60, 8

for epoch in range(MAX_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, grad_clip=5.0, device=device)
    dev_metrics, _ = evaluate(model, dev_loader, device)
    history.append({"epoch": epoch, "train_loss": train_loss, "dev_loss": dev_metrics["nll"], "dev_rmse": dev_metrics["rmse"]})
    if dev_metrics["nll"] < best_loss:
        best_loss, best_epoch, epochs_without_improvement = dev_metrics["nll"], epoch, 0
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": model.state_dict(), "params": {"stats": stats}, "epoch": epoch}, checkpoint_path)
        flag = "  <- new best, checkpoint saved"
    else:
        epochs_without_improvement += 1
        flag = f"  ({epochs_without_improvement} without improvement)"
    print(f"epoch {epoch:2d}  train {train_loss:7.3f}  dev {dev_metrics['nll']:7.3f}  dev rmse {dev_metrics['rmse']:5.2f} mm/yr{flag}")
    if epochs_without_improvement >= PATIENCE:
        print(f"early stop: {PATIENCE} epochs without a new best dev loss")
        break
print("best epoch:", best_epoch)

In [ ]:
import matplotlib.pyplot as plt
h = pl.DataFrame(history)
plt.plot(h["epoch"], h["train_loss"], label="train"); plt.plot(h["epoch"], h["dev_loss"], label="dev")
plt.axvline(best_epoch, ls="--", c="grey", label="best epoch"); plt.xlabel("epoch"); plt.ylabel("NLL"); plt.legend(); plt.show()

In [ ]:
# evaluate_on_dev: reload the BEST epoch (not the last one) and compare train vs dev, and dev per region / orbit
checkpoint = torch.load(checkpoint_path, map_location=device)
print("checkpoint keys:", list(checkpoint), "| saved at epoch", checkpoint["epoch"])
model.load_state_dict(checkpoint["state_dict"])

train_metrics, _ = evaluate(model, train_loader, device)
dev_metrics, dev_predictions = evaluate(model, dev_loader, device)
print("train:", {k: round(v, 3) for k, v in train_metrics.items()})
print("dev:  ", {k: round(v, 3) for k, v in dev_metrics.items()})

err = pl.col("mu_mm") - pl.col("y_mm")
dev_predictions.group_by("region").agg(pl.len().alias("n"), (err ** 2).mean().sqrt().alias("rmse"), err.abs().mean().alias("mae"),
                                       (err.abs() <= 1.645 * pl.col("sigma_mm")).mean().alias("inside_90"))

## Where to go from here

- Rerun section 5 with `USE_GSE = False` (section 3) and check that `x` has 3 instead of 67 columns.
- Change `K` (section 4.3) and watch the in-degree and the number of edges.
- Make the toy pools sparser (fewer points, larger spread in `make_graph_file`) and see how the edge distances and the
  dev metrics change.
- Set `hidden_dim`, `n_layers`, `heads` in section 7 and compare the parameter count.